# 🖥️ Which RunPod GPU should I use?

This notebook loads **google/gemma-4-E4B-it** (~8 B stored params, ~4 B "effective"). The VRAM you need depends on the **precision** you pick in the selector below — choose a pod that clears the *VRAM* column:

| Precision | Weights | ~VRAM (with overhead) | Recommended RunPod GPU | ~Price / hr |
|:---|:---|:---|:---|:---|
| **4-bit** (NF4) | ~4 GB | ~6 GB | **RTX A5000 · 24 GB** | **\$0.27** |
| **16-bit** (bf16) | ~16 GB | ~18–20 GB | **RTX A5000 · 24 GB** | **\$0.27** |
| **32-bit** (fp32) | ~32 GB | ~32–34 GB | **RTX A6000 · 48 GB** (or A40 48 GB) | **\$0.49** (A40 \$0.44) |

_Prices: RunPod Community Cloud, on-demand, ~July 2026 — they change often, check [runpod.io/pricing](https://www.runpod.io/pricing). Also set the pod's **Container/Volume disk to ≥ 100 GB**._

> **Note:** `gemma-4-E4B-it` is a full-precision, instruction-tuned model — the **E4B** means ~4 B *effective* parameters (8 B stored, via Per-Layer Embeddings), **not** 4-bit. Google also ships separately quantized `…-qat-*` variants if you want true 4-bit weights out of the box.

# Install libraries

In [1]:
%pip install torch
%pip install --upgrade transformers
%pip install bitsandbytes
%pip install accelerate
%pip install peft
%pip install datasets
%pip install evaluate
%pip install trl
%pip install matplotlib
%pip install tensorboard
%pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


...

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
for i in range(torch.cuda.device_count()):
   print(torch.cuda.get_device_properties(i).name)

NVIDIA RTX A6000


In [ ]:
from huggingface_hub import login

API_TOKEN = "os.environ.get(\"HF_TOKEN\")"
login(token=API_TOKEN)

# ⚙️ Choose precision -- run ONE of the three cells below

Run **exactly one** of the 4-bit / 16-bit / 32-bit cells. It sets `PRECISION` and `MODEL_KWARGS`,
which every section below uses (model loading, the on-disk folder `SAVED_MODEL_GEMMA4_E4B_<precision>`, and the
Hub repo names `my-base-gemma4-E4B-<precision>-...`). To switch precision later, run a different cell,
then re-run the **Load the model** cell.

In [3]:
# --- 4-bit (bitsandbytes NF4) ---
import torch
from transformers import BitsAndBytesConfig

PRECISION = "4bit"
MODEL_KWARGS = dict(
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    ),
    dtype=torch.bfloat16,
    device_map="cuda",
)
print("Selected precision:", PRECISION)

Selected precision: 4bit


In [2]:
# --- 16-bit (half precision, bfloat16) ---
import torch

PRECISION = "16bit"
MODEL_KWARGS = dict(
    dtype=torch.bfloat16,
    device_map="cuda",
)
print("Selected precision:", PRECISION)

Selected precision: 16bit


In [2]:
# --- 32-bit (full precision, float32) ---
import torch

PRECISION = "32bit"
MODEL_KWARGS = dict(
    dtype=torch.float32,
    device_map="cuda",
)
print("Selected precision:", PRECISION)

Selected precision: 32bit


# Base model from HUB

**What this notebook does:** download the original `google/gemma-4-E4B-it` from the Hugging Face Hub at the precision you selected, verify it generates a coherent answer, then persist it — save a copy to local disk **and** push a copy to your own Hub repo.

```mermaid
flowchart LR
    ORIG["🤗 Original Hub<br/>gemma-4-E4B-it"] -->|load| BM["base_model"]
    BM -->|test| OK["✅ generates"]
    BM -->|save| DISK["💾 disk<br/>SAVED_MODEL_&lt;prec&gt;"]
    BM -->|push| HUB["📦 Your Hub<br/>…-from-hub"]
    style ORIG fill:#2e7d32,color:#fff,stroke:#1b5e20
```

### Load the model

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Silence transformers' verbose warnings.
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

model_name = "google/gemma-4-E4B-it"

# Names derived from the precision you selected above -- used by every section below.
SAVED_DIR      = f"SAVED_MODEL_GEMMA4_E4B_{PRECISION}"
REPO_FROM_HUB  = f"lukaskellerstein/my-base-gemma4-E4B-{PRECISION}-from-hub"
REPO_FROM_DISK = f"lukaskellerstein/my-base-gemma4-E4B-{PRECISION}-from-disk"

base_model = AutoModelForCausalLM.from_pretrained(model_name, **MODEL_KWARGS)
base_tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Loaded {model_name}  |  precision={PRECISION}  |  hub repo={REPO_FROM_HUB}")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Loaded google/gemma-4-E4B-it  |  precision=4bit  |  hub repo=lukaskellerstein/my-base-gemma4-E4B-4bit-from-hub


Log model and tokenizer

In [5]:
def describe(model, tokenizer):
    """Print key facts about a loaded model + tokenizer. Reused in every section below."""
    print("--- Model ---")
    print("Type:", type(model))
    print("Architecture:", model)
    # Gemma 4 is multimodal, so the text vocab lives under the text sub-config.
    # get_text_config() returns that sub-config (and just returns the config itself
    # for plain text models), so this line works regardless of model family.
    print("Vocab size:", model.config.get_text_config().vocab_size)
    print("Input embeddings:", model.get_input_embeddings())
    print("Output embeddings:", model.get_output_embeddings())
    print("--- Tokenizer ---")
    print("Type:", type(tokenizer))
    print("Special tokens:", tokenizer.special_tokens_map)
    print("All tokens count:", len(tokenizer))
    print("Padding side:", tokenizer.padding_side)


describe(base_model, base_tokenizer)

--- Model ---
Type: <class 'transformers.models.gemma4.modeling_gemma4.Gemma4ForConditionalGeneration'>
Architecture: Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear4bit(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear4bit(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear4bit(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear4bit(in_features=768, out_features=768, bias=False)
      

### Test the model

#### via Pipeline

In [6]:
from transformers import pipeline

PROMPT = "What is AutoGen in abstract?"


def test_pipeline(model, tokenizer, prompt=PROMPT):
    """Test a model with a text-generation pipeline. Reused in every section below."""
    my_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

    # gemma-4-E4B-it is instruction-tuned, so we pass a chat (list of messages)
    # instead of a raw string. The pipeline then auto-applies the tokenizer's chat
    # template, giving a coherent answer (a raw string would ignore the template and ramble).
    messages = [{"role": "user", "content": prompt}]
    result = my_pipeline(messages, max_new_tokens=256, return_full_text=False)
    print(result[0]["generated_text"])


test_pipeline(base_model, base_tokenizer)

In abstract terms, **AutoGen is a framework designed to facilitate the creation and orchestration of multi-agent systems that can collaboratively solve complex tasks.**

Here is a breakdown of what that means in an abstract sense:

### Core Concept: Delegation and Collaboration

At its heart, AutoGen is not a single AI model; it is an **architecture** or a **system enabler**. It allows you to move beyond the paradigm of a single, monolithic prompt/response interaction and instead create a *system* where multiple, specialized AI entities interact with each other to achieve a high-level goal.

### Abstract Components Explained

1. **Agents (The Actors):**
    * **What they are:** These are independent, goal-oriented entities (which can be powered by LLMs). Each agent is assigned a *role* (e.g., Coder, Critic, Planner, Researcher, Reviewer).
    * **Their function:** They possess a specific set of skills, constraints, and objectives. They don't necessarily know everything; they know *what

#### via Model and Tokenizer

In [7]:
def test_generate(model, tokenizer, prompt=PROMPT):
    """Test a model by manually applying the chat template + generate. Reused below."""
    # Instruction-tuned model -> build inputs from a chat via the tokenizer's chat template.
    # add_generation_prompt=True appends the `<start_of_turn>model` marker so Gemma
    # actually answers instead of continuing the user's turn.
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to(model.device)

    # Generate, then decode only the answer.
    result = model.generate(**model_inputs, max_new_tokens=256)
    print(tokenizer.decode(result[0], skip_special_tokens=True))


test_generate(base_model, base_tokenizer)

user
What is AutoGen in abstract?
model
## AutoGen in Abstract: A Multi-Agent Conversation Framework for AI Workflow Orchestration

**AutoGen is a cutting-edge, open-source framework that enables the seamless orchestration of multiple Large Language Models (LLMs) to solve complex, multi-faceted problems through conversational collaboration.**

Instead of treating an LLM as a singular prompt-response entity, AutoGen models the problem as a **collaborative team simulation**. It allows developers to define various "agents"—each imbued with a specific **role, capabilities, and defined goal**—that interact with each other and the user through a structured, automated conversation flow.

Here is a breakdown of the key abstract concepts:

---

### 1. Core Paradigm: Multi-Agent System (MAS) Simulation
The fundamental abstraction is moving from a **single-turn interaction** ($\text{Input} \rightarrow \text{LLM} \rightarrow \text{Output}$) to a **multi-turn, iterative dialogue loop** ($\text{Inpu

#### via Tool call (single turn)

In [8]:
# Single-turn tool calling: pass `tools` to the chat template. An instruction-tuned
# model should answer with a tool call (function name + JSON arguments) instead of prose.
STOCK_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current price of a stock.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The ticker symbol for the stock, e.g. GOOG",
                    }
                },
                "required": ["ticker"],
            },
        },
    }
]


def test_tool_call(model, tokenizer, prompt, tools, max_new_tokens=256):
    """Single-turn tool calling: pass `tools`; the model should emit a tool call. Reused below."""
    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)

    # Greedy decoding -> deterministic tool call. Keep special tokens so the
    # tool-call markers stay visible.
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_call(base_model, base_tokenizer, "What is the current price of MSFT?", STOCK_TOOLS)


<|tool_call>call:get_stock_price{ticker:<|"|>MSFT<|"|>}<tool_call|><eos>


#### via Conversation with tool calls

In [6]:
def test_tool_conversation(model, tokenizer, max_new_tokens=256):
    """Multi-turn conversation that already contains an assistant tool call + a tool
    response, then a fresh user turn. Confirms the chat template round-trips the
    `tool_calls` / `tool` roles, then answers the latest turn. Reused below."""
    conversation = [
        {"role": "user", "content": "What's the weather like in Paris?"},
        {
            "role": "assistant",
            "content": "",
            "tool_calls": [
                {
                    "id": "call12345",
                    "type": "function",
                    "function": {
                        "name": "get_current_weather",
                        "arguments": '{"location": "Paris, France", "format": "celsius"}',
                    },
                }
            ],
        },
        {"role": "tool", "content": '{"content": 22}', "tool_call_id": "call12345"},
        {
            "role": "assistant",
            "content": "The current temperature in Paris, France is 22 degrees Celsius.",
        },
        {"role": "user", "content": "What's the weather like in San Francisco right now?"},
    ]

    weather_tools = [
        {
            "type": "function",
            "function": {
                "name": "get_current_weather",
                "description": "Get the current weather",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "The city and state, e.g. San Francisco, CA",
                        },
                        "format": {
                            "type": "string",
                            "enum": ["celsius", "fahrenheit"],
                            "description": "The temperature unit to use. Infer this from the users location.",
                        },
                    },
                    "required": ["location", "format"],
                },
            },
        }
    ]

    model_inputs = tokenizer.apply_chat_template(
        conversation, tools=weather_tools, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to(model.device)
    print("Tokenized conversation input_ids:")
    print(model_inputs["input_ids"])

    # Answer the latest user turn (should be another get_current_weather tool call for SF).
    result = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = result[0][model_inputs["input_ids"].shape[1]:]
    print("\nModel response:")
    print(tokenizer.decode(gen, skip_special_tokens=False))


test_tool_conversation(base_model, base_tokenizer)


Tokenized conversation input_ids:
tensor([[     2,    105,   9731,    107,     46, 163688, 236787,    828, 236779,
           4002, 236779,  19323, 236782,   7777, 236787,     52,   3407,    506,
           1873,   7606,     52, 236764,  19031,  29616,  15921,  29616,   5939,
          29616,   7777, 236787,     52,    818,   4022,   4360,    531,   1161,
         236761, 118936,    672,    699,    506,   5089,   4563, 236761,     52,
         236764,  24626,  24845,     52, 236755,  42024,     52, 236764,     52,
          46435,  15882,     52,   1604,   2084, 236787,     52,  35410,     52,
           1263,   7125,  29616,   7777, 236787,     52,    818,   3207,    532,
           1883, 236764,    545, 236761, 236759, 236761,   5054,  14322, 236764,
           8253,     52, 236764,   2084, 236787,     52,  35410,     52,   5237,
          15979,  24845,     52,   7125,     52, 236764,     52,   5939,     52,
           1604,   2084, 236787,     52,  60688,     52,   1807,     47,   


Model response:
<|tool_call>call:get_current_weather{{"location": "San Francisco, CA", "format": "fahrenheit"}}<tool_call|><eos>


### Save the model (to disk)

In [10]:
base_model.save_pretrained(SAVED_DIR)

base_tokenizer.save_pretrained(SAVED_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('SAVED_MODEL_GEMMA4_E4B_4bit/tokenizer_config.json',
 'SAVED_MODEL_GEMMA4_E4B_4bit/chat_template.jinja',
 'SAVED_MODEL_GEMMA4_E4B_4bit/tokenizer.json')

### Push the model (to HUB)

In [6]:
# push_to_hub() -> save_pretrained(save_original_format=True) can raise NotImplementedError
# in transformers 5.x, so save with save_original_format=False and upload the folder instead.
from huggingface_hub import upload_folder, create_repo

tmp = f"PUSH_TMP_{PRECISION}_from_hub"
create_repo(REPO_FROM_HUB, token=API_TOKEN, exist_ok=True)
base_model.save_pretrained(tmp, save_original_format=False)
base_tokenizer.save_pretrained(tmp)
upload_folder(repo_id=REPO_FROM_HUB, folder_path=tmp, token=API_TOKEN)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/lukaskellerstein/my-base-gemma4-E4B-32bit-from-hub/commit/5a49e30bb4ece8a368456d68030b54b542cbfcfe', commit_message='Upload folder using huggingface_hub', commit_description='', oid='5a49e30bb4ece8a368456d68030b54b542cbfcfe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/lukaskellerstein/my-base-gemma4-E4B-32bit-from-hub', endpoint='https://huggingface.co', repo_type='model', repo_id='lukaskellerstein/my-base-gemma4-E4B-32bit-from-hub'), pr_revision=None, pr_num=None)